# Validacion con codificadores humanos — analisis completo



## 0. Configuracion y carga

In [ ]:
N_PERMUTACIONES = 5000
SEMILLA = 20260301

try:
    from google.colab import files
    subidos = files.upload()
    RUTA = list(subidos.keys())[0]
except ImportError:
    RUTA = "Appendix_A3_Human_Validation_and_Ranking.xlsx"

print("Fichero:", RUTA)

In [ ]:
try:
    import krippendorff
except ImportError:
    %pip install krippendorff --quiet
    import krippendorff

import pandas as pd
import numpy as np
from scipy import stats

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
rng = np.random.default_rng(SEMILLA)

def wilson(exitos, n, conf=0.95):
    if n == 0:
        return (np.nan, np.nan)
    z = stats.norm.ppf(1 - (1 - conf) / 2)
    p = exitos / n
    den = 1 + z**2 / n
    c = (p + z**2 / (2*n)) / den
    h = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / den
    return (max(0, c - h) * 100, min(1, c + h) * 100)

def alfa(matriz, nivel):
    try:
        return krippendorff.alpha(reliability_data=matriz, level_of_measurement=nivel)
    except Exception:
        return np.nan

NOMBRES = {"Baseline [No Authority]": "Baseline s/aut", "Text B": "Priming",
           "Text C": "Bizarre", "Text D": "Change",
           "Text E": "Confirmation", "Text F": "Flaws"}
print("Listo")

### Categorias del codebook

In [ ]:
CORE_CATS = ["Yes, all/almost all preserved",
             "Partly; some changed/omitted",
             "No; important claims changed/omitted",
             "Uncertain"]

OMIS_CATS = ["No", "Yes, minor", "Yes, material", "Uncertain"]

COMP_CATS = {1: "Clearly not comparable", 2: "Mostly not comparable",
             3: "Partly comparable",      4: "Largely comparable",
             5: "Essentially equivalent"}

RANK_CATS = {1: "highest/most", 6: "lowest/least"}

# Solo las tres primeras de cada lista tienen orden de grado.
# Uncertain no ocupa lugar en la escala: es "no lo se", no un grado intermedio.
CORE_ORDEN = {"Yes, all/almost all preserved": 2,
              "Partly; some changed/omitted": 1,
              "No; important claims changed/omitted": 0}
OMIS_ORDEN = {"No": 0, "Yes, minor": 1, "Yes, material": 2}

NO_CONFORME = "(no conforme al codebook)"
print("Core:", len(CORE_CATS), "categorias   Omision:", len(OMIS_CATS),
      "categorias   Comparabilidad:", len(COMP_CATS), "puntos")

In [ ]:
s1 = pd.read_excel(RUTA, sheet_name="Stage1_Validation")
s1 = s1.rename(columns={
    "Coder ID": "coder", "Topic": "topic", "Modified Text ID": "texto",
    "Core factual claims preserved?": "core",
    "Omission/contradiction?": "omision",
})
s1 = s1.rename(columns={c: "comp" for c in s1.columns
                        if str(c).startswith("Overall informational comparability")})
s1 = s1[s1["coder"].notna()]
s1 = s1[s1["core"].notna() | s1["comp"].notna()]

s1["tema"] = np.where(s1["topic"].str.startswith("Resp"), "Virus", "Inflacion")
s1["version"] = s1["tema"] + " / " + s1["texto"].map(NOMBRES)

print(f"Codificadores con datos : {s1['coder'].nunique()}  {sorted(s1['coder'].unique())}")
print(f"Versiones evaluadas     : {s1['version'].nunique()}")
print(f"Filas                   : {len(s1)}")

In [ ]:

def a_categoria(valor, categorias):
    "Devuelve una categoria del codebook, None si falta, o NO_CONFORME."
    if pd.isna(valor):
        return None
    s = str(valor).strip().lower()
    for c in categorias:
        if s == c.lower():                       # etiqueta completa
            return c
    # mitad de una etiqueta partida por la coma; solo vale si es inequivoca
    candidatas = [c for c in categorias
                  if c.partition(", ")[1]
                  and s in (c.partition(", ")[0].lower(), c.partition(", ")[2].lower())]
    if len(candidatas) == 1:
        return candidatas[0]
    # "Yes" a secas en Omission encaja con "Yes, minor" y con "Yes, material".
    # Resuelto a "Yes, minor" tras revisar el texto libre de las tres filas
    # afectadas (decision del equipo, documentada en metodos).
    if s == "yes" and set(candidatas) == {"Yes, minor", "Yes, material"}:
        return "Yes, minor"
    # Variantes de redaccion de la primera categoria de core claims
    # ("Yes, all preserved", "Yes, almost all preserved"): misma categoria.
    if "preserved" in s and "Yes, all/almost all preserved" in categorias:
        if s.startswith("no"):
            return "No; important claims changed/omitted"
        if s.startswith("partly") or s.startswith("partial"):
            return "Partly; some changed/omitted"
        return "Yes, all/almost all preserved"
    return NO_CONFORME

s1["core_lab"] = s1["core"].map(lambda v: a_categoria(v, CORE_CATS))
s1["omis_lab"] = s1["omision"].map(lambda v: a_categoria(v, OMIS_CATS))

for nombre, orig, lab in [("Core factual claims preserved?", "core", "core_lab"),
                          ("Omission/contradiction?", "omision", "omis_lab")]:
    print("=" * 78)
    print(nombre, "— texto recibido  ->  categoria del codebook")
    print("=" * 78)
    t = s1.groupby(lab)[orig].value_counts().rename("n").reset_index()
    print(t.to_string(index=False))
    print(f"  no conformes: {int((s1[lab] == NO_CONFORME).sum())}"
          f"   sin respuesta: {int(s1[lab].isna().sum())}")
    print()





---
# Stage 1 · A. Core factual claims preserved



In [ ]:
filas = []
for v, g in s1.groupby("version"):
    c = g["core_lab"].value_counts()
    fila = {"version": v}
    for cat in CORE_CATS:
        fila[cat.split(";")[0].split(",")[0]] = int(c.get(cat, 0))
    fila["no conf."] = int(c.get(NO_CONFORME, 0))
    n_val = sum(int(c.get(cat, 0)) for cat in CORE_ORDEN)
    n_si = int(c.get("Yes, all/almost all preserved", 0))
    lo, hi = wilson(n_si, n_val)
    fila.update({"n_valido": n_val,
                 "% preservado": round(100*n_si/n_val, 1) if n_val else np.nan,
                 "IC inf": round(lo, 1), "IC sup": round(hi, 1)})
    filas.append(fila)

core_tab = pd.DataFrame(filas).sort_values("% preservado", ascending=False)
display(core_tab.reset_index(drop=True))
print("Columnas Yes / Partly / No / Uncertain = las cuatro categorias del codebook")
print("% preservado = categoria 'Yes, all/almost all preserved' sobre n_valido")

In [ ]:
s1["core_num"] = s1["core_lab"].map(CORE_ORDEN)
M_core = s1.pivot_table(index="coder", columns="version",
                        values="core_num", aggfunc="first").values
print(f"alfa ordinal (Yes > Partly > No) : {alfa(M_core,'ordinal'): .3f}")
print(f"alfa nominal (categorias sueltas): {alfa(M_core,'nominal'): .3f}")
print(f"respuestas usadas                : {int(np.isfinite(M_core).sum())}")

---
# Stage 1 · B. Omission / contradiction



In [ ]:
filas = []
for v, g in s1.groupby("version"):
    c = g["omis_lab"].value_counts()
    fila = {"version": v}
    for cat in OMIS_CATS:
        fila[cat] = int(c.get(cat, 0))
    fila["no conf."] = int(c.get(NO_CONFORME, 0))
    n_val = sum(int(c.get(cat, 0)) for cat in OMIS_ORDEN)
    n_mat = int(c.get("Yes, material", 0))
    lo, hi = wilson(n_mat, n_val)
    fila.update({"n_valido": n_val,
                 "% material": round(100*n_mat/n_val, 1) if n_val else np.nan,
                 "IC inf": round(lo, 1), "IC sup": round(hi, 1)})
    filas.append(fila)

omis_tab = pd.DataFrame(filas).sort_values("% material", ascending=False)
display(omis_tab.reset_index(drop=True))
print("Columnas No / Yes, minor / Yes, material / Uncertain = las cuatro del codebook")
print("Las tres respuestas que llegaron como 'Yes' a secas se asignan a")
print("'Yes, minor' tras revisar su texto libre. Decision documentada.")

In [ ]:
s1["omis_num"] = s1["omis_lab"].map(OMIS_ORDEN)
M_omis = s1.pivot_table(index="coder", columns="version",
                        values="omis_num", aggfunc="first").values
print(f"alfa ordinal (No < minor < material): {alfa(M_omis,'ordinal'): .3f}")
print(f"alfa nominal (categorias sueltas)   : {alfa(M_omis,'nominal'): .3f}")
print(f"respuestas usadas                   : {int(np.isfinite(M_omis).sum())}")

---
# Stage 1 · C. Overall informational comparability




In [ ]:
fuera = s1[~s1["comp"].isin(COMP_CATS)]["comp"].dropna()
if len(fuera):
    print("AVISO, valores fuera de la escala 1-5:", sorted(fuera.unique()))

mat = s1.pivot_table(index="version", columns="coder", values="comp", aggfunc="first")
mat = mat.reindex(mat.mean(axis=1).sort_values(ascending=False).index)
display(mat)

reparto = pd.DataFrame({f"{k} {v}": (mat == k).sum(axis=1) for k, v in COMP_CATS.items()})
reparto["media"] = mat.mean(axis=1).round(2)
reparto["mediana"] = mat.median(axis=1)
reparto["% 4 o 5"] = (100 * (mat >= 4).mean(axis=1)).round(0)
display(reparto)

In [ ]:
M_comp = mat.T.values
print(f"alfa ordinal  : {alfa(M_comp,'ordinal'): .3f}")
print(f"alfa interval : {alfa(M_comp,'interval'): .3f}")
print()

X = mat.dropna(axis=1).values
n_items, n_coders = X.shape
gm = X.mean()
MS_items  = n_coders * ((X.mean(1) - gm)**2).sum() / (n_items - 1)
MS_coders = n_items  * ((X.mean(0) - gm)**2).sum() / (n_coders - 1)
MS_error  = ((X - X.mean(1, keepdims=True) - X.mean(0, keepdims=True) + gm)**2).sum() \
            / ((n_items - 1) * (n_coders - 1))

icc_uno   = (MS_items - MS_error) / (MS_items + (n_coders-1)*MS_error
                                     + n_coders*(MS_coders - MS_error)/n_items)
icc_media = (MS_items - MS_error) / (MS_items + (MS_coders - MS_error)/n_items)
F = MS_items / MS_error
gl1, gl2 = n_items - 1, (n_items - 1) * (n_coders - 1)
p_icc = 1 - stats.f.cdf(F, gl1, gl2)
nivel = lambda v: ("pobre" if v < .50 else "aceptable" if v < .75
                   else "bueno" if v < .90 else "excelente")

print(f"ICC(2,1)  un codificador suelto : {icc_uno:.3f}  ({nivel(icc_uno)})")
print(f"ICC(2,k)  media de los {n_coders}       : {icc_media:.3f}  ({nivel(icc_media)})")
print(f"F({gl1},{gl2}) = {F:.2f}, p = {p_icc:.4f}"
      f"  ->  {'las versiones difieren' if p_icc < .05 else 'no concluyente'}")

pares = igual = cerca = 0
for fila in X:
    for i in range(len(fila)):
        for j in range(i+1, len(fila)):
            pares += 1
            igual += fila[i] == fila[j]
            cerca += abs(fila[i] - fila[j]) <= 1
print(f"\nAcuerdo bruto: misma categoria {100*igual/pares:.0f} %, "
      f"a un punto o menos {100*cerca/pares:.0f} % de {pares} pares")

---
# Stage 2 · Rankings


In [ ]:
s2 = pd.read_excel(RUTA, sheet_name="Stage2_Rankings")
s2 = s2.rename(columns={"Coder ID": "coder", "Topic": "topic",
                        "Authority condition": "autoridad", "Task": "tarea"})
cols_rank = [c for c in s2.columns if str(c).startswith("Text ") and "rank" in str(c)]
MAPEO = {"Text A": "Baseline", "Text B": "Priming", "Text C": "Bizarre",
         "Text D": "Change", "Text E": "Confirmation", "Text F": "Flaws"}
RANGOS_VALIDOS = [1, 2, 3, 4, 5, 6]

def estado(fila):
    v = sorted(int(x) for x in fila[cols_rank].dropna())
    if not v:                    return "vacia"
    if len(v) < 6:               return "incompleta"
    if v == RANGOS_VALIDOS:      return "valida"
    return "rango repetido o fuera de 1-6"

s2["estado"] = s2.apply(estado, axis=1)
display(pd.crosstab(s2["coder"], s2["estado"]))

malas = s2[s2["estado"] == "rango repetido o fuera de 1-6"]
if len(malas):
    def detalle(f):
        v = [int(x) for x in f[cols_rank].dropna()]
        return pd.Series({"repite": sorted({x for x in v if v.count(x) > 1}),
                          "falta": [x for x in RANGOS_VALIDOS if x not in v]})
    print("Filas a corregir:")
    display(pd.concat([malas[["coder","topic","autoridad","tarea"]],
                       malas.apply(detalle, axis=1)], axis=1).reset_index(drop=True))

In [ ]:
val = s2[s2["estado"] == "valida"].copy()
val["set_id"] = (val["coder"] + " | " + val["topic"] + " | "
                 + val["autoridad"] + " | " + val["tarea"])
largo = val.melt(id_vars=["set_id","coder","topic","autoridad","tarea"],
                 value_vars=cols_rank, var_name="col", value_name="rango")
largo["condicion"] = largo["col"].str.replace(" rank", "", regex=False).map(MAPEO)
largo["rango"] = largo["rango"].astype(int)

print(f"Sets validos: {largo['set_id'].nunique()} de {len(s2)}")
print(f"Codificadores utilizables: {largo['coder'].nunique()}")
display(largo.pivot_table(index="tarea", columns="condicion",
                          values="rango", aggfunc="mean").round(2))
print("Bajo intercambiabilidad el rango medio esperado de cada condicion es 3.5")
print("(menor que 3.5 = mejor valorada, mayor = peor valorada)")

In [ ]:
def test_permutacion(df, n_perm=N_PERMUTACIONES):
    conds = sorted(df["condicion"].unique())
    esperado = (len(conds) + 1) / 2
    M = np.array([g.sort_values("condicion")["rango"].values
                  for _, g in df.groupby("set_id")])
    T_obs = ((M.mean(0) - esperado) ** 2).sum()
    idx = rng.random((n_perm,) + M.shape).argsort(axis=2)
    perm = np.take_along_axis(np.broadcast_to(M, (n_perm,) + M.shape), idx, axis=2)
    T_null = ((perm.mean(1) - esperado) ** 2).sum(1)
    return T_obs, (np.sum(T_null >= T_obs) + 1) / (n_perm + 1), M.shape[0]

filas = []
for t, g in largo.groupby("tarea"):
    T, p, n = test_permutacion(g)
    filas.append({"tarea": t, "T": round(T, 2), "p": round(p, 4), "sets": n})
res = pd.DataFrame(filas)

o = np.argsort(res["p"].values)
r = np.empty_like(o); r[o] = np.arange(1, len(res) + 1)
adj = res["p"].values * len(res) / r
res["p_BH"] = np.minimum.accumulate(adj[o][::-1])[::-1][np.argsort(o)].round(4)
display(res)
print("p pequeno = los humanos NO ordenan al azar: la condicion importa")

---
# Resumen

In [ ]:
print("STAGE 1")
print(f"  codificadores            : {s1['coder'].nunique()} de 10")
print(f"  alfa core claims         : {alfa(M_core,'ordinal'): .3f} ordinal /"
      f" {alfa(M_core,'nominal'): .3f} nominal")
print(f"  alfa omision             : {alfa(M_omis,'ordinal'): .3f} ordinal /"
      f" {alfa(M_omis,'nominal'): .3f} nominal")
print(f"  alfa comparabilidad      : {alfa(M_comp,'ordinal'): .3f} ordinal")
print(f"  ICC(2,1) / ICC(2,k)      : {icc_uno:.3f} / {icc_media:.3f}")
print()
print("  Mas cambio material (categoria 'Yes, material'):")
for _, r_ in omis_tab.head(3).iterrows():
    print(f"    {r_['version']:28s} {r_['% material']:5.1f} %"
          f"  IC [{r_['IC inf']:.0f}, {r_['IC sup']:.0f}]")
print()
print("  Mejor comparabilidad:")
for v, r_ in reparto.head(3).iterrows():
    print(f"    {v:28s} media {r_['media']:.2f}  mediana {r_['mediana']:.0f}")
print()
print("STAGE 2")
print(f"  sets validos             : {largo['set_id'].nunique()} de {len(s2)}")
print(f"  codificadores utilizables: {largo['coder'].nunique()}")
print(f"  dimensiones no aleatorias: {(res['p_BH'] < .05).sum()} de {len(res)}")